In [ ]:
# guaranteed_fix.py (CORRECTED)
import os
import shutil
from pathlib import Path
from tqdm import tqdm

def create_guaranteed_working_dataset():
    """Create a dataset structure that's guaranteed to work with YOLOv8."""
    
    print("Creating guaranteed working dataset structure...")
    
    # New clean directory
    new_root = Path("/home/ika1/yzlm/Re-id/object_detection_Re-ID/WP_YOLO_Clean")
    new_root.mkdir(exist_ok=True)
    
    # Create the exact structure YOLOv8 expects
    (new_root / "images").mkdir(exist_ok=True)
    (new_root / "labels").mkdir(exist_ok=True)
    
    # Source paths
    src_root = Path("/home/ika1/yzlm/Re-id/object_detection_Re-ID/WiderPersonYOLO/WiderPerson")
    src_lists_dir = Path("/home/ika1/yzlm/Re-id/object_detection_Re-ID/WiderPersonYOLO/WP_YOLO")
    src_images_dir = src_root / "Images"
    
    # --- Part 1: Symlink labels and their corresponding images ---
    print("\nCopying images that have labels...")
    label_files = list((src_root / "labels").glob("*.txt"))
    for label_file in tqdm(label_files, desc="Processing Labeled Data"):
        img_name = label_file.stem + ".jpg"
        src_img = src_images_dir / img_name
        
        if src_img.exists():
            dst_img = new_root / "images" / img_name
            dst_label = new_root / "labels" / label_file.name
            
            if not dst_img.exists():
                dst_img.symlink_to(src_img)
            if not dst_label.exists():
                dst_label.symlink_to(label_file)

    # ========================================================================= #
    # START OF THE FIX: Ensure ALL images from splits exist, especially test set
    # ========================================================================= #
    print("\nEnsuring all images from train/val/test splits are symlinked...")
    for split in ['train', 'val', 'test']:
        split_list_path = src_lists_dir / f"{split}.txt"
        if not split_list_path.exists():
            print(f"Warning: Source list {split_list_path} not found.")
            continue

        with open(split_list_path, 'r') as f:
            for line in tqdm(f, desc=f"Checking {split} images"):
                img_name = Path(line.strip()).name
                if not img_name:
                    continue
                
                src_img_path = src_images_dir / img_name
                dst_img_path = new_root / "images" / img_name
                
                # If the destination link doesn't exist yet, create it.
                # This is crucial for the label-less test images.
                if not dst_img_path.exists() and src_img_path.exists():
                    dst_img_path.symlink_to(src_img_path)
    # ========================================================================= #
    # END OF THE FIX
    # ========================================================================= #

    # --- Part 3: Create the new list files with relative paths ---
    for split in ['train', 'val', 'test']:
        print(f"\nProcessing {split} split list file...")
        src_list = src_lists_dir / f"{split}.txt"
        dst_list = new_root / f"{split}.txt"
        
        with open(src_list, 'r') as f_in, open(dst_list, 'w') as f_out:
            for line in f_in:
                if line.strip():
                    img_name = Path(line.strip()).name
                    # Write relative path from dataset root
                    f_out.write(f"./images/{img_name}\n")
    
    # --- Part 4: Create the YAML file ---
    yaml_content = f"""# WiderPerson dataset - YOLOv8 format
path: {new_root.resolve()}
train: train.txt
val: val.txt  
test: test.txt

# Classes
nc: 1
names: ['person']
"""
    
    yaml_path = new_root / "dataset.yaml"
    with open(yaml_path, 'w') as f:
        f.write(yaml_content)
    
    print(f"\nDataset created at: {new_root}")
    print(f"YAML file: {yaml_path}")
    
    # --- Part 5: Verify Final Structure ---
    print("\nVerifying final structure:")
    total_images_in_splits = sum(1 for split in ['train', 'val', 'test'] for line in open(src_lists_dir / f'{split}.txt') if line.strip())
    print(f"Images in new dataset: {len(list((new_root / 'images').glob('*.jpg')))} files (should be close to {total_images_in_splits})")
    print(f"Labels in new dataset: {len(list((new_root / 'labels').glob('*.txt')))} files")
    
    return str(yaml_path)

# Run it
if __name__ == "__main__":
    yaml_path = create_guaranteed_working_dataset()
    
    # Test immediately (You can keep this part)
    print("\n" + "="*50)
    print("TESTING NEW DATASET (val set)")
    print("="*50)
    
    from ultralytics import YOLO
    
    model = YOLO("yolov8n.pt")
    metrics = model.val(
        data=yaml_path,
        imgsz=640,
        batch=8,
        device=0
    )
    
    print(f"\n✅ Results:")
    print(f"mAP@50: {metrics.box.map50:.4f}")
    print(f"mAP@50-95: {metrics.box.map:.4f}")


Creating guaranteed working dataset structure...

Copying images that have labels...


Processing Labeled Data: 100%|██████████| 9000/9000 [00:00<00:00, 14575.68it/s]



Ensuring all images from train/val/test splits are symlinked...


Checking train images: 8000it [00:00, 48488.15it/s]
Checking val images: 1000it [00:00, 45137.41it/s]
Checking test images: 4382it [00:00, 23341.88it/s]



Processing train split list file...

Processing val split list file...

Processing test split list file...

Dataset created at: /home/ika1/yzlm/Re-id/object_detection_Re-ID/WP_YOLO_Clean
YAML file: /home/ika1/yzlm/Re-id/object_detection_Re-ID/WP_YOLO_Clean/dataset.yaml

Verifying final structure:
Images in new dataset: 13382 files (should be close to 13382)
Labels in new dataset: 9000 files

TESTING NEW DATASET (val set)
Ultralytics 8.3.168 🚀 Python-3.11.11 torch-2.6.0+cu124 CUDA:0 (NVIDIA GeForce RTX 3050 Laptop GPU, 3768MiB)
YOLOv8n summary (fused): 72 layers, 3,151,904 parameters, 0 gradients, 8.7 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1595.7±578.3 MB/s, size: 36.9 KB)


val: Scanning /home/ika1/yzlm/Re-id/object_detection_Re-ID/WP_YOLO_Clean/labels... 1000 images, 0 backgrounds, 0 corrupt: 100%|██████████| 1000/1000 [00:00<00:00, 2101.51it/s]

val: New cache created: /home/ika1/yzlm/Re-id/object_detection_Re-ID/WP_YOLO_Clean/labels.cache



                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 125/125 [00:09<00:00, 13.71it/s]


                   all       1000      27353      0.718      0.504      0.587      0.242
                person       1000      27353      0.718      0.504      0.587      0.242
Speed: 0.3ms preprocess, 4.8ms inference, 0.0ms loss, 1.6ms postprocess per image
Results saved to runs/detect/val16

✅ Results:
mAP@50: 0.5873
mAP@50-95: 0.2424
